# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide to loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing record sets and fields by their `@id` values for precision and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL: 
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Date published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Let's review the available record sets, fields, and their `@id`s as defined by the Croissant schema. All references to data entities will use their `@id` for consistency.

### Explore Available Record Sets
We will inspect the record sets, printing their `@id` values and key metadata.

In [ ]:
# List all available record sets and their field `@id`s
print("Available record sets in this dataset:")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- Record set @id: {record_set['@id']}")
    if 'name' in record_set:
        print(f"  Name: {record_set['name']}")
    if 'description' in record_set:
        print(f"  Description: {record_set['description']}")
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            # Each field is a dict with '@id', 'name', etc.
            if isinstance(field, dict):
                print(f"    - {field['@id']} ({field.get('name', '')})")
            else:
                print(f"    - {field}")
    print()
# For future referencing, get a list of record set ids:
record_set_ids = [rs['@id'] for rs in record_sets]

### Preview Records
View a sample of records from a chosen record set. 
Replace `<record_set_id>` below with one of the actual record set `@id` values from the previous cell.

In [ ]:
# Print sample records from the first record set (by @id)
if record_set_ids:
    record_set_id = record_set_ids[0]
    print(f"Sample records from record set @id: {record_set_id}")
    sample = []
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        sample.append(record)
        if i >= 2:
            break
    for rec in sample:
        print(rec)
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load the data from a specific record set into a DataFrame for analysis.

We'll demonstrate this using the first available record set's `@id` as an example. All field references use their `@id`.

In [ ]:
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded DataFrame for record set @id: {rsid} with shape: {df.shape}")

# For demonstration, pick the first DataFrame if available
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    print(f"Columns for @id {chosen_record_set_id}:")
    print(dataframes[chosen_record_set_id].columns.tolist())
    dataframes[chosen_record_set_id].head()
else:
    print("No dataframes extracted.")

## 4. Exploratory Data Analysis (EDA)
Now that we have loaded the records into a DataFrame, let's perform basic EDA, referencing fields by their `@id`.

- We will filter records using a numeric field (choose one, e.g., protein abundance or molecular weight). 
- Normalize the numeric field.
- Optionally, group by a relevant annotation/category field.

Replace `<numeric_field_id>` and `<group_field_id>` below with field `@id`s (not names) identified from earlier overview.

In [ ]:
# Example configuration (replace with actual field `@id`s as appropriate):
numeric_field_id = None
group_field_id = None
# Inspect columns/field ids
if dataframes:
    df = dataframes[chosen_record_set_id]
    print("Column (field @id) candidates:", df.columns.tolist())
    # Try to auto-select a likely numeric field (e.g., 'cr:protein_abundance', 'cr:mw', etc.)
    # For demonstration choose first column with numeric dtype
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Try grouping by the second column if it's not numeric
    for col in df.columns:
        if col != numeric_field_id:
            group_field_id = col
            break

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean()  # as an example filter
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Grouping
        if group_field_id:
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
                print(grouped_df.head())
    else:
        print("No numeric field detected for analysis.")
else:
    print("No dataframe to perform EDA.")

## 5. Visualization
Visualize distributions or relationships between the fields using matplotlib or seaborn.

This example creates a histogram of the selected numeric field and a boxplot grouped by another field. Field references use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    df = dataframes[chosen_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook we have loaded, explored, and visualized the FAIR² mass spectrometry dataset using `mlcroissant`, referencing all data entities by their unique `@id`. This workflow enables precise, reproducible FAIR data processing.

- **Record sets** and **fields** are referenced by `@id`.
- Dataframes can be extracted for each record set and analyzed using common data science techniques.
- This approach supports future-oriented, schema-driven analytics for machine actionable data.

For additional analysis, use the `mlcroissant` documentation to inspect more metadata and relationships:
`https://mlcommons.github.io/croissant/python-api.html`